# B3 ? Formula-Based CVaR-Constrained A-S Parameter PPO

**Purpose:** Use the same formula-based A-S parameter policy as B1/B2, but train with a CVaR drawdown constraint. PPO remains the optimizer because the policy action is continuous `[u_gamma, u_skew]` and the current Lagrangian implementation is on-policy.


In [ ]:
import sys
import pathlib
import shutil
from dataclasses import replace

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = next(
    (p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
     if (p / "procs").exists()),
    pathlib.Path.cwd(),
)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import VecNormalize

from procs.gym.experiment_config import ReplayExperimentConfig
from procs.gym.data_loader import load_multi_day_with_features
from procs.gym.calibration import calibrate_from_arrays
from procs.gym.formula_as import FORMULA_FEATURE_NAMES
from procs.gym.sb3_wrapper import StableBaselinesTradingEnvironment
from procs.gym.notebook_support import (
    build_formula_multi_day_replay_env,
    calibrate_cvar_threshold_sampled_windows,
    make_vecnorm,
    evaluate_formula_rl_per_day,
)
from procs.rewards import CjMmCriterion
from procs.gym.cvar_lagrangian import (
    DrawdownCostWrapper,
    CVaRLagrangianCallback,
)
from procs.gym.helpers.plotting import plot_cvar_training

cfg = ReplayExperimentConfig()
cfg.ensure_artifact_dirs()
print(f"Repo root : {cfg.repo_root}")


## Section 1 — Data Loading and Train/Test Split

In [ ]:
TRAIN_DAYS = 6

daily_S, daily_dt, dates, daily_features = load_multi_day_with_features(
    str(cfg.datasets_dir),
    pair=cfg.pair,
    tick_size=cfg.tick_size,
)

train_S        = daily_S[:TRAIN_DAYS]
train_dt       = daily_dt[:TRAIN_DAYS]
train_dates    = dates[:TRAIN_DAYS]
train_features = daily_features[:TRAIN_DAYS]

test_S         = daily_S[TRAIN_DAYS:]
test_dt        = daily_dt[TRAIN_DAYS:]
test_dates     = dates[TRAIN_DAYS:]
test_features  = daily_features[TRAIN_DAYS:]

EXPECTED_TEST_DAYS = 23
if len(train_S) != TRAIN_DAYS or len(test_S) != EXPECTED_TEST_DAYS:
    raise ValueError(
        f"Expected {TRAIN_DAYS} train days and {EXPECTED_TEST_DAYS} test days, "
        f"found {len(train_S)} and {len(test_S)}."
    )
if set(map(str, train_dates)) & set(map(str, test_dates)):
    raise ValueError("Train/test date overlap detected.")

param_rows = []
for date, S, dt in zip(train_dates, train_S, train_dt):
    sigma, A, kappa = calibrate_from_arrays(S, dt, tick_size=cfg.tick_size)
    param_rows.append({"Day": date, "sigma": sigma, "A": A, "kappa": kappa})

df_train_params = pd.DataFrame(param_rows).set_index("Day")
sigma_train = float(df_train_params["sigma"].median())
A_train = float(df_train_params["A"].median())
kappa_train = float(df_train_params["kappa"].median())
cfg = replace(cfg, A=A_train, kappa=kappa_train)

FORMULA_KWARGS = dict(
    sigma=sigma_train,
    gamma_min=0.01,
    gamma_max=0.9,
    skew_ticks_max=5.0,
)

train_snapshots = sum(len(S) for S in train_S)

print(f"Training: {len(train_S)} separate days, {train_snapshots:,} total snapshots")
print(f"Test    : {len(test_S)} days ({test_dates[0]} -> {test_dates[-1]})")
print("Train-only A-S market parameters:")
print(f"  sigma={sigma_train:.8f}, A={A_train:.6f}, kappa={kappa_train:.2f}")
print(f"Formula features: {list(FORMULA_FEATURE_NAMES)}")


## Section 2 — Load B1 and B2 Best Models

In [ ]:
# Load B1 (unconstrained PPO) for CVaR threshold calibration
model_b1 = PPO.load(str(cfg.model_path("ppo_b1_doge")), device="cpu")
print(f"Loaded B1: {cfg.model_path('ppo_b1_doge')}.zip")

# Load best B2 alpha for warm-start
best_alpha_path = cfg.result_path("b2_best_alpha.txt")
with open(best_alpha_path) as f:
    best_alpha = float(f.read().strip())
print(f"Best B2 α = {best_alpha}  (from {best_alpha_path})")

model_b2_best = PPO.load(str(cfg.model_path(f"ppo_b2_alpha{best_alpha}_doge")), device="cpu")
print(f"Loaded B2 best: ppo_b2_alpha{best_alpha}_doge.zip")

## Section 3 — CVaR Threshold Calibration

The threshold `d` is calibrated from the B1 agent's drawdown on **training days only**.
Using windowed calibration (`n_steps=2048`) matches the window size used in training.

`d = CVaR_{0.2}(MaxDD_{B1, window}) × (1 − 0.2)`

**Critical:** No test-day data is used here.

In [ ]:
# Sample PPO-length windows across all training days for B3 threshold calibration.
# This keeps the threshold training-only without reintroducing artificial day joins.
print("Calibrating CVaR threshold from formula-AS B1 rollouts on sampled training windows ...")
d, cvar_uc = calibrate_cvar_threshold_sampled_windows(
    daily_midprices=train_S,
    daily_dt_arrays=train_dt,
    daily_market_features=train_features,
    model=model_b1,
    vecnorm_path=cfg.vecnorm_path("vecnorm_b1"),
    config=cfg,
    n_steps=cfg.ppo_n_steps,
    cvar_alpha=0.2,
    n_windows=50,
    tighten=0.2,
    seed=cfg.evaluation_seed,
    verbose=True,
    sigma=sigma_train,
    formula_gamma_min=FORMULA_KWARGS["gamma_min"],
    formula_gamma_max=FORMULA_KWARGS["gamma_max"],
    formula_skew_ticks_max=FORMULA_KWARGS["skew_ticks_max"],
)

print()
print(f"CVaR threshold d = {d:.6f}  (raw CVaR = {cvar_uc:.6f})")

thresh_path = cfg.result_path("cvar_threshold_d.txt")
with open(thresh_path, "w") as f:
    f.write(f"{d}\n{cvar_uc}\n")
print(f"Saved -> {thresh_path}")


## Section 4 — B3 Training

Env stack: `VecNormalize(DrawdownCostWrapper(StableBaselinesTradingEnvironment(env)))`

- `DrawdownCostWrapper` reads `self.env.env.model_dynamics.state` (i.e., the raw `TradingEnvironment`) — this is why VecNormalize wraps the outside, not inside.
- The callback holds a direct reference to `cost_wrapper` to access `step_costs` regardless of wrapping.
- Warm-started from B2 best weights via `set_parameters()`.

**`TOTAL_TIMESTEPS` guide:**
- Quick local test: `200_000`
- Full Snellius: `max(2 * len(S_train), 2_000_000)`

In [ ]:
TOTAL_TIMESTEPS = max(train_snapshots, 1_000_000)
CVAR_ALPHA = 0.2
LR_LAMBDA = 0.01
LAMBDA_MAX = 500.0

print(f"CVaR threshold d  = {d:.6f}")
print(f"alpha (tail prob) = {CVAR_ALPHA}")
print(f"lambda learning rate = {LR_LAMBDA}")
print(f"lambda max = {LAMBDA_MAX}")
print(f"Total timesteps = {TOTAL_TIMESTEPS:,}")

train_env_b3 = build_formula_multi_day_replay_env(
    train_S,
    train_dt,
    cfg,
    daily_market_features=train_features,
    reward_fn=CjMmCriterion(per_step_inventory_aversion=cfg.phi),
    mode="sequential",
    **FORMULA_KWARGS,
)
train_sb3_b3 = StableBaselinesTradingEnvironment(train_env_b3)
cost_wrapper = DrawdownCostWrapper(train_sb3_b3)
train_vn_b3 = make_vecnorm(cost_wrapper, cfg, training=True, norm_reward=False)

cvar_callback = CVaRLagrangianCallback(
    cost_wrapper=cost_wrapper,
    cvar_alpha=CVAR_ALPHA,
    dd_threshold=d,
    lr_lambda=LR_LAMBDA,
    lambda_max=LAMBDA_MAX,
    verbose=1,
)

model_b3 = PPO(
    "MlpPolicy",
    train_vn_b3,
    **cfg.ppo_kwargs(),
    tensorboard_log=str(cfg.repo_root / "tb_logs" / "b3"),
    verbose=1,
    device="cpu",
)

# Warm-start from formula-AS B2 best ? keeps the constrained optimization near a feasible policy.
model_b3.set_parameters(model_b2_best.get_parameters())
print("B3 warm-started from formula-AS B2 best weights.")

model_b3.learn(total_timesteps=TOTAL_TIMESTEPS, callback=cvar_callback)


In [ ]:
model_zip = cfg.model_path("ppo_b3_doge").with_suffix(".zip")
model_archive = cfg.model_path("ppo_b3_doge_direct_depth").with_suffix(".zip")
if model_zip.exists() and not model_archive.exists():
    shutil.copy2(model_zip, model_archive)
    print(f"Archived previous direct-depth model -> {model_archive}")

vecnorm_path = cfg.vecnorm_path("vecnorm_b3")
vecnorm_archive = cfg.vecnorm_path("vecnorm_b3_direct_depth")
if vecnorm_path.exists() and not vecnorm_archive.exists():
    shutil.copy2(vecnorm_path, vecnorm_archive)
    print(f"Archived previous direct-depth VecNormalize -> {vecnorm_archive}")

model_b3.save(str(cfg.model_path("ppo_b3_doge")))
train_vn_b3.save(str(cfg.vecnorm_path("vecnorm_b3")))
print(f"Saved formula-AS model   -> {cfg.model_path('ppo_b3_doge')}.zip")
print(f"Saved formula-AS vecnorm -> {cfg.vecnorm_path('vecnorm_b3')}")


## Section 5 — λ Convergence Diagnostics

Healthy convergence: λ rises from 0, stabilises at a non-zero value well below `lambda_max`.
- λ → 0: constraint not active (threshold too loose)
- λ → λ_max: constraint too tight or training CVaR still above threshold

In [ ]:
if cvar_callback.lambda_history and cvar_callback.cvar_history:
    try:
        plot_cvar_training(
            cvar_history=cvar_callback.cvar_history,
            lambda_history=cvar_callback.lambda_history,
            dd_threshold=d,
        )
    except Exception:
        # Fallback manual plot if plot_cvar_training API differs
        fig, axes = plt.subplots(1, 2, figsize=(13, 4))
        rollouts = range(len(cvar_callback.lambda_history))

        axes[0].plot(rollouts, cvar_callback.cvar_history, label=f"CVaR_{CVAR_ALPHA}")
        axes[0].axhline(d, color="crimson", linestyle="--", label=f"threshold d={d:.4f}")
        axes[0].set_title("Training CVaR(MaxDD)")
        axes[0].set_xlabel("Rollout")
        axes[0].legend()
        axes[0].grid(True, alpha=0.3)

        axes[1].plot(rollouts, cvar_callback.lambda_history, color="darkorange", label="λ")
        axes[1].axhline(LAMBDA_MAX, color="gray", linestyle=":", alpha=0.7, label=f"λ_max={LAMBDA_MAX}")
        axes[1].set_title("Lagrange Multiplier λ")
        axes[1].set_xlabel("Rollout")
        axes[1].legend()
        axes[1].grid(True, alpha=0.3)

        plt.suptitle("B3 CVaR Lagrangian — Training Convergence", fontsize=13)
        plt.tight_layout()
        plt.savefig(str(cfg.result_path("b3_lambda_convergence.png")), dpi=150, bbox_inches="tight")
        plt.show()

    print(f"Final λ = {cvar_callback.lambda_history[-1]:.4f}  "
          f"(λ_max={LAMBDA_MAX}, {'OK' if cvar_callback.lambda_history[-1] < LAMBDA_MAX else 'AT CAP'})")
    print(f"Final CVaR = {cvar_callback.cvar_history[-1]:.6f}  (threshold d={d:.6f})")
else:
    print("No callback history recorded — check TOTAL_TIMESTEPS > ppo_n_steps.")


In [ ]:
# Save callback history for nb4
import json
history_path = cfg.result_path("b3_callback_history.json")
with open(history_path, "w") as f:
    json.dump({
        "lambda_history": cvar_callback.lambda_history,
        "cvar_history":   cvar_callback.cvar_history,
        "dd_threshold":   d,
        "lambda_max":     LAMBDA_MAX,
        "cvar_alpha":     CVAR_ALPHA,
    }, f)
print(f"Saved callback history → {history_path}")


## Section 6 — Test Evaluation (Days 7–29)

In [ ]:
model_b3_loaded = PPO.load(str(cfg.model_path("ppo_b3_doge")), device="cpu")

df_b3 = evaluate_formula_rl_per_day(
    model=model_b3_loaded,
    vecnorm_path=cfg.vecnorm_path("vecnorm_b3"),
    test_S=test_S,
    test_dt=test_dt,
    test_dates=test_dates,
    test_market_features=test_features,
    config=cfg,
    seed=cfg.evaluation_seed,
    num_rollouts=cfg.evaluation_rollouts,
    **FORMULA_KWARGS,
)

out_path = cfg.result_path("b3_test_results.csv")
archive_path = cfg.result_path("b3_direct_depth_test_results.csv")
if out_path.exists() and not archive_path.exists():
    shutil.copy2(out_path, archive_path)
    print(f"Archived previous direct-depth results -> {archive_path}")

df_b3.to_csv(out_path)
print(f"Saved formula-AS B3 results -> {out_path}")
print(f"Evaluated {len(df_b3)} test days with {cfg.evaluation_rollouts} rollouts per day.")


## Section 7 — B2 vs B3 Comparison (Central Thesis Result)

In [ ]:
try:
    df_b2_best = pd.read_csv(
        cfg.result_path(f"b2_alpha{best_alpha}_test_results.csv"), index_col="Day"
    )
    b2_available = True
except FileNotFoundError:
    print("B2 results not found — run nb2 first.")
    b2_available = False

pd.set_option("display.float_format", "{:.4f}".format)
METRICS = ["Sharpe", "Sortino", "Max DD", "P&L-to-MAP", "Final PnL"]

print("=== B3 Summary ===")
print(pd.DataFrame({"Mean": df_b3[METRICS].mean(), "Std": df_b3[METRICS].std()}).T.to_string())

if b2_available:
    print(f"\n=== B2 (α={best_alpha}) Summary ===")
    print(pd.DataFrame({"Mean": df_b2_best[METRICS].mean(), "Std": df_b2_best[METRICS].std()}).T.to_string())


In [ ]:
if b2_available:
    days = range(len(df_b3))
    x = np.arange(len(df_b3))
    width = 0.38

    fig, axes = plt.subplots(1, 2, figsize=(15, 5))

    # Plot 1: MaxDD per day
    axes[0].bar(x - width/2, df_b2_best["Max DD"].values, width, label=f"B2 α={best_alpha}", color="steelblue", alpha=0.8)
    axes[0].bar(x + width/2, df_b3["Max DD"].values,      width, label="B3 CVaR",            color="seagreen",  alpha=0.8)
    axes[0].axhline(d, color="crimson", linestyle="--", label=f"CVaR threshold d={d:.4f}")
    axes[0].set_title("Max Drawdown per Test Day\nB2 (reward shaping) vs B3 (CVaR constraint)")
    axes[0].set_xlabel("Test day index")
    axes[0].set_ylabel("Max Drawdown")
    axes[0].legend(fontsize=9)
    axes[0].grid(True, alpha=0.3, axis="y")

    # Plot 2: Sharpe per day
    axes[1].bar(x - width/2, df_b2_best["Sharpe"].values, width, label=f"B2 α={best_alpha}", color="steelblue", alpha=0.8)
    axes[1].bar(x + width/2, df_b3["Sharpe"].values,      width, label="B3 CVaR",            color="seagreen",  alpha=0.8)
    axes[1].axhline(0, color="black", linewidth=0.8)
    axes[1].set_title("Sharpe Ratio per Test Day\nB2 vs B3")
    axes[1].set_xlabel("Test day index")
    axes[1].set_ylabel("Sharpe")
    axes[1].legend(fontsize=9)
    axes[1].grid(True, alpha=0.3, axis="y")

    plt.tight_layout()
    plt.savefig(str(cfg.result_path("b2_vs_b3_comparison.png")), dpi=150, bbox_inches="tight")
    plt.show()

    b3_better_dd = (df_b3["Max DD"] < df_b2_best["Max DD"]).sum()
    print(f"\nB3 achieves lower MaxDD on {b3_better_dd}/{len(df_b3)} test days.")
    print(f"B3 mean MaxDD = {df_b3['Max DD'].mean():.4f}  vs  B2 mean MaxDD = {df_b2_best['Max DD'].mean():.4f}")
